# Data Preparation Pipeline

This notebook prepares data for the scoring model.

## Steps
1. Load raw data from SQL Server
2. Clean and transform
3. Calculate RFM metrics
4. Export for scoring

In [ ]:
from scipy.stats import linregress
from datetime import date
import datetime
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
from statistics import variance as var
from scipy import stats
from numpy import convolve
from dateutil.relativedelta import relativedelta
!pip install openpyxl
import re
import sys
import os
import csv

In [ ]:
########### update the date and cmd20 based on execution date
###################### cmd5 sch_demo_02.table_01, sch_demo_03.table_02, sch_demo_03.table_03
###################### cmd7 sch_demo_03.table_06, /dbfs/mnt/demo/dl_demo_ds/pipeline/dictionaries/CRM_Contacts.csv
###################### cmd9 sch_demo_05.table_07
###################### cmd11 sch_demo_01.table_08b
###################### cmd13 sch_demo_06.table_09
###################### cmd42 sch_demo_07.table_10

currdt = datetime.date(2023, 1, 31)

def eom(dt,rangemt):
    from dateutil.relativedelta import relativedelta
    mt = dt - relativedelta(months=rangemt)
    import calendar
    import datetime
    mt = datetime.date(mt.year,mt.month,calendar.monthrange(mt.year, mt.month)[1])
    return mt



In [ ]:

jdbcUrl_01 = "jdbc:sqlserver://demo001.corp.example.com:1433;databaseName=db_demo;SchemaName=sch_demo_01".format("demo001.corp.example.com", 1433, "db_demo")

jdbcUrl_02 = "jdbc:sqlserver://demo001.corp.example.com:1433;databaseName=db_demo;SchemaName=sch_demo_02".format("demo001.corp.example.com", 1433, "db_demo")

jdbcUrl_03 = "jdbc:sqlserver://demo001.corp.example.com:1433;databaseName=db_demo;SchemaName=sch_demo_03".format("demo001.corp.example.com", 1433, "db_demo")

jdbcUrl_04 = "jdbc:sqlserver://demo001.corp.example.com:1433;databaseName=db_demo;SchemaName=sch_demo_04".format("demo001.corp.example.com", 1433, "db_demo")
connectionProperties = {
  "user" : "",
  "password" : ""}# need to replace

print(" Demo Data Preparation Script is in Execution")
print(" -------------------------------------------")
print(" Demo Input sheet is for: ",currdt.strftime('%d-%b-%Y').upper() )

## Paths ###
base_path = '/dbfs/mnt/demo/dl_demo_ds/pipeline' 
dic_path = '/dbfs/mnt/demo/dl_demo_ds/pipeline/dictionaries'
Log_path = '/dbfs/mnt/demo/dl_demo_ds/pipeline/logs'



In [ ]:
### Logging scripts after each print statements ###
###################################################
import pytz
IST = pytz.timezone('Asia/Kolkata')
batch_start = datetime.datetime.now(IST)
script_log= []
log_1 = '### Demo Dataprep Log Report ### :\n\n1. Script Started at: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)

In [ ]:
###################### to exclusion list add three ids
pushdown_query = "(SELECT distinct(GlobalID) from sch_demo_02.table_01) temp" 
exclude2 = spark.read.jdbc(url=jdbcUrl_02, table=pushdown_query, properties=connectionProperties)
exclude2 = exclude2.toPandas()
exclude2.columns = ['contact_global_id_exclude']
# exclude2['contact_global_id_exclude'] = exclude2['contact_global_id_exclude'].astype(int)
exclude2['contact_global_id_exclude'] = exclude2['contact_global_id_exclude'].astype(str)
exclude2['exclude'] = 1

pushdown_query = "(SELECT distinct([Contact.Global Id]) from sch_demo_03.table_02) temp" 
exclude1 = spark.read.jdbc(url=jdbcUrl_03, table=pushdown_query, properties=connectionProperties)
exclude1 = exclude1.toPandas()
exclude1.columns = ['contact_global_id_exclude']
exclude1['contact_global_id_exclude'] = exclude1['contact_global_id_exclude'].astype(str)
exclude1['exclude'] = 1

exclude = pd.concat([exclude1,exclude2],axis=0)
exclude.reset_index(inplace=True,drop=True)
exclude.loc[len(exclude.index)] = [1000001,1]
exclude.loc[len(exclude.index)] = [1000002,1]
exclude.loc[len(exclude.index)] = [1000003,1]
exclude.rename(columns={'contact_global_id_exclude':'Contact_Global_ID'},inplace=True)

################ to house accounts add two ids
pushdown_query = "(SELECT * from sch_demo_03.table_03) temp" 
house_accounts = spark.read.jdbc(url=jdbcUrl_03, table=pushdown_query, properties=connectionProperties)
house_accounts = house_accounts.toPandas()
house_accounts = pd.DataFrame(house_accounts['CONTACT_GLOBAL_ID'].unique(),columns=['CONTACT_GLOBAL_ID'])
house_accounts['ind'] = 1
house_accounts.reset_index(inplace=True,drop=True)
house_accounts.loc[len(house_accounts.index)] = [1000004,1]
# house_accounts.loc[len(house_accounts.index)] = [1000005,1]
house_accounts['CONTACT_GLOBAL_ID']=house_accounts['CONTACT_GLOBAL_ID'].astype('str')
house_accounts.rename(columns={'CONTACT_GLOBAL_ID':'Contact_Global_ID'},inplace=True)
house_accounts.drop_duplicates(inplace=True)



In [ ]:
def remove_liq_funds(df):
  df['FUND_NAME1'] = df.FUND_NAME.str.lower()
  df['FUND_NAME2'] = df.FUND_NAME1.str.contains('money|liquid', regex=True)
  df = df.loc[df.FUND_NAME2 == False,] 
  df.drop(['FUND_NAME1','FUND_NAME2'],axis=1,inplace=True)
  return df

def remove_exclusion_ids(df):
  df = pd.merge(df,exclude,on= 'Contact_Global_ID',how='left')
  df = df[df['exclude'] != 1.0]
  del df['exclude']
  return df

def remove_house_accounts(df):
  df = pd.merge(df,house_accounts,how='left',on='Contact_Global_ID')
  df = df[df['ind'] != 1]
  del df['ind']
  return df

def remove_unmapped_ids(df):
  df['len'] = df['Contact_Global_ID'].apply(lambda x:len(x))
  df = df[df['len'].isin([7,8])]
  del df['len']
  df.reset_index(inplace=True,drop=True)
  return df


In [ ]:
pushdown_query = "(SELECT * from sch_demo_03.table_04) temp" 
crm_contacts = spark.read.jdbc(url=jdbcUrl_03, table=pushdown_query, properties=connectionProperties)
crm_contacts = crm_contacts.toPandas()
crm_contacts = crm_contacts[crm_contacts.Date_inserted== crm_contacts.Date_inserted.max()]
crm_contacts=crm_contacts[crm_contacts['Primary Owner'].isin(['Surname01, Name01','Surname02, Name02','Surname03, Name03','Surname04, Name04'])]
crm_contacts = crm_contacts[['Contact ID']]
crm_contacts.columns = ['Contact_Global_ID']
crm_contacts.Contact_Global_ID=crm_contacts.Contact_Global_ID.astype('str')
print('msd contacts: {0}'.format(crm_contacts.shape))

crm_contacts = remove_exclusion_ids(crm_contacts)
print('msd contacts w/o exclusion ids: {0}'.format(crm_contacts.shape))

crm_contacts = remove_house_accounts(crm_contacts)
print('msd contacts w/o house accounts: {0}'.format(crm_contacts.shape))



In [ ]:
# ### ######### AUM Correction ####################
# ############################# Only for Latest AUM 
# ############# Assets data ##################
# ###################################
# from pyspark.sql.functions import to_date
# pushdown_query = "(SELECT * from sch_demo_01.table_08 where geo_flag = 'Segment_01' and SALES_OFFICE = 'Region_01' and psp in ('Name01 Surname01','Name03 Surname03','Name02 Surname02') ) temp" 
# assets_table = spark.read.jdbc(url=jdbcUrl_01, table=pushdown_query, properties=connectionProperties)

# # assets_table = assets_table.select('*', to_date(assets_table.asset_date_asof).alias('date'))
# # assets_table = assets_table[assets_table.date == currdt] # Only for latest Data 

# assets_table = assets_table.toPandas()

# funds = pd.read_csv(dic_path+"/fundname_std.csv")
# assets_table = assets_table.merge(funds, left_on = 'FUND_NAME', right_on = 'fund_name',how='left')

# assets_table = remove_exclusion_ids(assets_table)
# assets_table = remove_liq_funds(assets_table)
# assets_table = remove_house_accounts(assets_table)
# assets_table = remove_unmapped_ids(assets_table)

# assets_table_mod = assets_table[['org','source', 'Contact_Global_ID','FUND_NAME', 'fund', 'FUND_OBJECTIVES', 'asset_date_asof','new_aum']]
# assets_table_mod.columns = ['org','source', 'contact_global_id','FUND_NAME', 'fund', 'FUND_OBJECTIVES', 'process_date','assets']

# assets_table_mod['contact_global_id'] = assets_table_mod['contact_global_id'].astype(str)
# assets_table_mod['contact_global_id'] = assets_table_mod['contact_global_id'].fillna('NULL')
# assets_table_mod['contact_global_id'] = assets_table_mod['contact_global_id'].apply(lambda x: 'NULL' if x == 'None' else x)

# assets_table_mod = assets_table_mod[assets_table_mod.contact_global_id!= 'NULL']
# assets_table_mod['assets'] = pd.to_numeric(assets_table_mod.assets)


# # ########aum snapshots and #funds in aum
# # assets_table_mod1 = pd.DataFrame(assets_table_mod.groupby(['contact_global_id'],as_index=False).agg({'assets':'sum'}))

# # funds_assets = pd.DataFrame(assets_table_mod.groupby('contact_global_id').FUND_NAME.nunique()).reset_index()
# # assets_table_mod1 = assets_table_mod1.merge(funds_assets,on='contact_global_id',how='outer') #(1711, 2) (1938, 5)

In [ ]:
######################## sales data
pushdown_query = "(SELECT * from sch_demo_01.table_08b where geo_flag= 'Segment_01' and SALES_OFFICE = 'Region_01' and psp in ('Name01 Surname01','Name03 Surname03','Name02 Surname02')) temp" 
sales_data = spark.read.jdbc(url=jdbcUrl_01, table=pushdown_query, properties=connectionProperties)
sales_data = sales_data.toPandas()
sales_data['Alloc_Sales_Period'] = sales_data['Alloc_Sales_Period'].apply(lambda x: float(x))
sales_data['Process_Date'] = sales_data['Process_Date'].apply(lambda x:datetime.datetime.strptime(x, '%Y-%m-%d').date())
# sales_data['Month'] = [datetime.date(a.year,a.month,1) for a in sales_data['Process_Date']]

sales_data['Contact_Global_ID'].fillna('null',inplace=True)
sales_data = sales_data[~(sales_data['Contact_Global_ID'].isin([' ','',0,'0','null']))]
sales_data.reset_index(drop=True, inplace=True)
print('transactions data pull: $ {0} Mn'.format((sales_data[(sales_data.Process_Date> currdt - relativedelta(months=12))&(sales_data.Process_Date<= currdt)].Alloc_Sales_Period.sum()/1000000).round(0)))

sales_data = remove_liq_funds(sales_data)
print('sales data w/o liquid funds: $ {0} Mn'.format((sales_data[(sales_data.Process_Date> currdt - relativedelta(months=12))&(sales_data.Process_Date<= currdt)].Alloc_Sales_Period.sum()/1000000).round(0)))

sales_data = remove_exclusion_ids(sales_data)
print('transactions data w/o exclusion ids: $ {0} Mn'.format((sales_data[(sales_data.Process_Date> currdt - relativedelta(months=12))&(sales_data.Process_Date<= currdt)].Alloc_Sales_Period.sum()/1000000).round(0)))

sales_data = remove_house_accounts(sales_data)
print('transactions data w/o branches: $ {0} Mn'.format((sales_data[(sales_data.Process_Date> currdt - relativedelta(months=12))&(sales_data.Process_Date<= currdt)].Alloc_Sales_Period.sum()/1000000).round(0)))

sales_data = remove_unmapped_ids(sales_data)
print('transactions data w/o unmapped ids: $ {0} Mn'.format((sales_data[(sales_data.Process_Date> currdt - relativedelta(months=12))&(sales_data.Process_Date<= currdt)].Alloc_Sales_Period.sum()/1000000).round(0)))

sales = sales_data.groupby(['Contact_Global_ID','Org', 'Process_Date','FUND_NAME']).Alloc_Sales_Period.sum().reset_index()
redemptions = sales_data.groupby(['Contact_Global_ID','Org', 'Process_Date','FUND_NAME']).Redemptions.sum().reset_index()
sales.Contact_Global_ID=sales.Contact_Global_ID.astype('str')
redemptions.Contact_Global_ID=redemptions.Contact_Global_ID.astype('str')

#remove sales <0, or redemptions >0 to make sure avregare values are not wrong
sales = sales[sales['Alloc_Sales_Period'] > 0]
sales.reset_index(inplace=True,drop=True)
redemptions = redemptions[redemptions['Redemptions'] < 0]
redemptions.reset_index(inplace=True,drop=True)

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n2. Sales Data processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 2. Demo Sales Data is in process: ProgressBar:10%")


## Data Loading

Loading data from multiple sources:
- Sales data
- Assets data
- Activity data

In [ ]:
print(" 1. Demo Assets Data is in process: ProgressBar:5%")

################## historic AUM data to filter out latest aum
asset_snapshot = spark.sql(""" select * from sch_demo_05.table_07 where geo_flag='Segment_01' and psp in ('Name01 Surname01','Name03 Surname03','Name02 Surname02')""")

df_assets_master = asset_snapshot[['org', 'Contact_Global_ID', 'asset_date_asof', 'FUND_NAME', 'unified_CONTACT_NAME', 'new_aum']].toPandas()
df_assets_master['new_aum'] = pd.to_numeric(df_assets_master['new_aum'])

df_assets_master = df_assets_master.groupby(['org', 'Contact_Global_ID', 'asset_date_asof', 'FUND_NAME']).new_aum.sum().reset_index()
df_assets_master['asset_date_asof'] = df_assets_master['asset_date_asof'].apply(lambda x:datetime.datetime.strptime(x, '%Y-%m-%d').date())

import calendar
import datetime
df_assets_master['Month'] = df_assets_master.asset_date_asof.apply(lambda x: datetime.date(x.year, x.month, calendar.monthrange(x.year, x.month)[1]))
df_assets_master.Contact_Global_ID=df_assets_master.Contact_Global_ID.astype('str')
print('assets data pull: $ {0} Mn'.format((df_assets_master[df_assets_master.Month == df_assets_master.Month.max()].new_aum.sum()/1000000).round(0)))

########### remove blank contact id
df_assets_master['Contact_Global_ID'].fillna('null',inplace=True)
df_assets_master = df_assets_master[~(df_assets_master['Contact_Global_ID'].isin([' ','',0,'0','null']))]
df_assets_master.reset_index(drop=True, inplace=True)
print('assets data pull: $ {0} Mn'.format((df_assets_master[df_assets_master.Month==df_assets_master.Month.max()].new_aum.sum()/1000000).round(0)))

df_assets_master = remove_exclusion_ids(df_assets_master)
print('assets data w/o exclusion ids: $ {0} Mn'.format((df_assets_master[df_assets_master.Month==df_assets_master.Month.max()].new_aum.sum()/1000000).round(0)))

df_assets_master = remove_liq_funds(df_assets_master)
print('assets data w/o liquid funds: $ {0} Mn'.format((df_assets_master[df_assets_master.Month==df_assets_master.Month.max()].new_aum.sum()/1000000).round(0)))

df_assets_master = remove_house_accounts(df_assets_master)
print('assets data w/o house accounts: $ {0} Mn'.format((df_assets_master[df_assets_master.Month==df_assets_master.Month.max()].new_aum.sum()/1000000).round(0)))

df_assets_master = remove_unmapped_ids(df_assets_master)
print('assets data w/o unmapped ids: $ {0} Mn'.format((df_assets_master[df_assets_master.Month==df_assets_master.Month.max()].new_aum.sum()/1000000).round(0)))


In [ ]:
# ################### impute LM aum for latest month based on Sales and Redemptions 
# lm_aum_data = df_assets_master[(df_assets_master['org'] == 'LM')&(df_assets_master['Month'] == eom(currdt,1))]
# lm_aum_data = lm_aum_data.groupby(['org','Contact_Global_ID','FUND_NAME']).new_aum.sum().reset_index()
# lm_aum_data.rename(columns={'org':'Org'},inplace=True)

# ########## one month sales
# sales_lm = sales[(sales['Process_Date'] >= currdt.replace(day=1)) & (sales['Process_Date'] <= currdt) & (sales['Org'] == 'LM')]
# sales_lm = sales_lm.groupby(['Org','Contact_Global_ID','FUND_NAME']).Alloc_Sales_Period.sum().reset_index()

# ########## one month redemption
# red_lm = redemptions[(redemptions['Process_Date'] >= currdt.replace(day=1)) & (redemptions['Process_Date'] <= currdt) & (redemptions['Org'] == 'LM')]
# red_lm = red_lm.groupby(['Org','Contact_Global_ID','FUND_NAME']).Redemptions.sum().reset_index()

# sales_red_lm = pd.merge(sales_lm, red_lm, how='outer', on=['Org','Contact_Global_ID','FUND_NAME'])
# sales_red_lm.fillna(0,inplace=True)

# aum_sales_red_lm = pd.merge(lm_aum_data, sales_red_lm, how='outer', on=['Org','Contact_Global_ID','FUND_NAME'])
# aum_sales_red_lm.fillna(0,inplace=True)
# aum_sales_red_lm['new_aum1'] = aum_sales_red_lm['new_aum'] + (aum_sales_red_lm['Alloc_Sales_Period'] + aum_sales_red_lm['Redemptions'])
# aum_sales_red_lm['asset_date_asof'] = currdt
# aum_sales_red_lm['Month'] = currdt
# aum_sales_red_lm = aum_sales_red_lm[['Org', 'Contact_Global_ID', 'asset_date_asof', 'FUND_NAME', 'new_aum1','Month']]
# aum_sales_red_lm.rename(columns={'new_aum1':'new_aum','Org':'org'},inplace=True)

# df_assets_master = df_assets_master[~(df_assets_master['org'] == 'LM')|~(df_assets_master['Month'] == currdt)].append(aum_sales_red_lm)

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n3. Assets Data processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)

In [ ]:
##################### activities data
activity_query = ("select distinct ACTIVITY_ID, ACTIVITYTYPE as ACTIVITY_TYPE, ACTIVITY_TYPE_NORM, SUBJECT as ACTIVITY_DESCRIPTION, ACTIVITY_START_DATE, TRIM(UPPER(EMPL_NAME)) as ACTIVITY_OWNER_NAME, ACTIVITY_NOTES as ACTIVITY_INSIGHT, CONTACT_ID, PARENT_FIRM_GLOBAL_ID as FIRM_ID, PRODUCT_NAME, UNIFIED_PRODUCT_STRATEGY as PRODUCT_STRATEGY, VALUE_ADDED_CONTACT_FLAG from sch_demo_08.table_12 where EMPL_NAME in ('Surname01, Name01','Surname02, Name02','Surname03, Name03','Surname04, Name04') ")
activity = spark.sql(activity_query)
activity = activity.toPandas()

# activity['ACTIVITY_START_DATE'] = activity['ACTIVITY_START_DATE'].apply(lambda x:datetime.datetime.strptime(x, '%Y-%m-%d').date())
activity = activity[['CONTACT_ID','ACTIVITY_START_DATE','FIRM_ID','ACTIVITY_TYPE_NORM']]
activity.columns = ['Contact_Global_ID','ACTIVITY_START_DATE', 'account_id','ACTIVITY_TYPE_NORM']
activity.dropna(subset = ['Contact_Global_ID'], inplace=True)
activity.Contact_Global_ID = activity.Contact_Global_ID.astype('str')
print('activity data pull: {0}'.format(activity[(activity.ACTIVITY_START_DATE > currdt - relativedelta(months=12))&(activity.ACTIVITY_START_DATE <= currdt)].ACTIVITY_TYPE_NORM.value_counts()))

activity = remove_exclusion_ids(activity)
print('activity data pull: {0}'.format(activity[(activity.ACTIVITY_START_DATE > currdt - relativedelta(months=12))&(activity.ACTIVITY_START_DATE <= currdt)].ACTIVITY_TYPE_NORM.value_counts()))

activity = remove_house_accounts(activity)
print('activity data pull: {0}'.format(activity[(activity.ACTIVITY_START_DATE > currdt - relativedelta(months=12))&(activity.ACTIVITY_START_DATE <= currdt)].ACTIVITY_TYPE_NORM.value_counts()))

activity = remove_unmapped_ids(activity)
print('activity data pull: {0}'.format(activity[(activity.ACTIVITY_START_DATE > currdt - relativedelta(months=12))&(activity.ACTIVITY_START_DATE <= currdt)].ACTIVITY_TYPE_NORM.value_counts()))


In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n4. Activity Data processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 4. Demo Activity Data is in process: ProgressBar:10%")


In [ ]:
####################### filtering data beyond current date
####################### Copy to Sales Master to calculate DV based on future sales
sales_master=sales.copy()
red_master=redemptions.copy()
df_assets_original = df_assets_master.copy()
activity_master = activity.copy()

sales = sales[sales['Process_Date'] <= currdt]
redemptions=redemptions[redemptions['Process_Date']<= currdt]
df_assets_master = df_assets_master[df_assets_master['asset_date_asof']<= currdt]
activity=activity[activity['ACTIVITY_START_DATE']<= currdt]

In [ ]:
############################# unique id's from msdynamics
crm_contacts1 = crm_contacts[['Contact_Global_ID']]
crm_contacts1.drop_duplicates(inplace=True)
contacts_base = crm_contacts1.copy()
print('contact base: {0}'.format(contacts_base.shape))


In [ ]:
########## sales[['Contact_Global_ID','Alloc_Sales_Period','Process_Date']]
def rfm_analysis(sales,start_date,end_date):
  
  RFM_master_f6m = sales[(sales['Process_Date'] > start_date) & (sales['Process_Date'] <= end_date)]
  RFM_master_f6m = pd.DataFrame(RFM_master_f6m['Contact_Global_ID'].unique(), columns=['Contact_Global_ID'])
  RFM_master_f6m = RFM_master_f6m.drop_duplicates()
  RFM_master_f6m.reset_index(inplace=True,drop=True)
  
  rfmf6m_new = sales[(sales['Process_Date'] > start_date) & (sales['Process_Date'] <= end_date)]
  rfmf6m_new.reset_index(drop=True,inplace=True)
  rfmf6m_new['freq'] = 1
  rfmf6m_new = rfmf6m_new[['Contact_Global_ID', 'Alloc_Sales_Period','Process_Date','freq']]
  rfmf6m_new.columns = ['Contact_Global_ID_new','Sales','Date','Freq']
  
  ################## frequency
  rfmf6m_new = rfmf6m_new.groupby(['Contact_Global_ID_new']).agg({'Date':'max','Sales':'sum','Freq':'sum'}).reset_index()
  rfmf6m_new = rfmf6m_new.drop_duplicates(subset=['Contact_Global_ID_new'])
  
  ################# recency
  rfmf6m_new['Date'] = rfmf6m_new['Date'].apply(lambda x: x if(pd.notnull(x)) else start_date + relativedelta(days=1))
  rfmf6m_new['current_date'] = end_date
  rfmf6m_new['days'] = 0
  
  def days(d1,d2):
      x = d1-d2
      return x.days
  
  rfmf6m_new['days'] = rfmf6m_new.apply(lambda x:days(x.current_date,x.Date),axis=1)
  rfmf6m_new.fillna(0,inplace=True)
  
  ############### monetory value
  rfmf6m_new['sales_new'] = rfmf6m_new['Sales']
  
  rfmf6m_new = rfmf6m_new[['Contact_Global_ID_new','days','Freq','sales_new']]
  rfmf6m_new.columns= ['Contact_Global_ID_new','Recency','Frequency','Monetary']
  rfmf6m_new.fillna(0,inplace=True)
  
  quartiles = rfmf6m_new.quantile(q=[0.333,0.667])
  quartiles=quartiles.to_dict()
  
  def RClass(x,p,d):
      if x < d[p][0.333]:
          return 3
      elif x >= d[p][0.333] and x < d[p][0.667]:
          return 2
      elif x >= d[p][0.667]:
          return 1
  
  def FClass(x,p,d):
      if x <= d[p][0.333]:
          return 1
      elif x > d[p][0.333] and x <= d[p][0.667]:
          return 2
      elif x > d[p][0.667]:
          return 3
  
  def MClass(x,p,d):
      if x <= d[p][0.333]:
          return 1
      elif x > d[p][0.333] and x < d[p][0.667]:
          return 2
      elif x >= d[p][0.667]:
          return 3
  
  rfmf6m_new = rfmf6m_new
  rfmf6m_new['R_Quartile'] = rfmf6m_new['Recency'].apply(RClass, args=('Recency',quartiles))
  rfmf6m_new['F_Quartile'] = rfmf6m_new['Frequency'].apply(FClass, args=('Frequency',quartiles))
  rfmf6m_new['M_Quartile'] = rfmf6m_new['Monetary'].apply(MClass, args=('Monetary',quartiles))
  
  rfmf6m_new['RFMClass'] = rfmf6m_new.R_Quartile.map(str) \
                              + rfmf6m_new.F_Quartile.map(str) \
                              + rfmf6m_new.M_Quartile.map(str)
  return rfmf6m_new 

In [ ]:
############################### RFM first 6 months
rfmSeg = rfm_analysis(sales, currdt- relativedelta(months=12), currdt - relativedelta(months=6))
rfm_f6 = rfmSeg[['Contact_Global_ID_new','RFMClass']]
rfm_f6.columns = ['Contact_Global_ID_new','RFM_f6']

############################### RFM second 6 months
rfmSeg = rfm_analysis(sales, currdt - relativedelta(months=6), currdt)
rfm_s6 = rfmSeg[['Contact_Global_ID_new','RFMClass']]
rfm_s6.columns = ['Contact_Global_ID_new','RFM_s6']

################# combined rfm f6 & s6 months
algorithm = pd.merge(rfm_f6,rfm_s6,on='Contact_Global_ID_new',how='outer')
algorithm.fillna(111,inplace=True)

################# Creating RFM Movement segments
algorithm['R'] = (pd.to_numeric(algorithm['RFM_s6'])/100).round(0) - (pd.to_numeric(algorithm['RFM_f6'])/100).round(0)
algorithm['F'] = ((pd.to_numeric(algorithm['RFM_s6'])/10)%10).round(0) - ((pd.to_numeric(algorithm['RFM_f6'])/10)%10).round(0)
algorithm['M'] = (pd.to_numeric(algorithm['RFM_s6'])%10).round(0) - (pd.to_numeric(algorithm['RFM_f6'])%10).round(0)
      
        
################# create movement variable from R,F,M
algorithm['Movement'] = ""
for i in range(len(algorithm)):
    if algorithm.loc[i,'R'] != '' and algorithm.loc[i,'F'] != '' and algorithm.loc[i,'M'] != '':
        if int(algorithm.loc[i,'R']) >= 0 and int(algorithm.loc[i,'F']) >= 0 and int(algorithm.loc[i,'M']) >= 0 and (int(algorithm.loc[i,'R'])+ int(algorithm.loc[i,'F'])+int(algorithm.loc[i,'M'])) > 0:
            algorithm.loc[i,'Movement'] = "gained"
        elif int(algorithm.loc[i,'R']) <= 0 and int(algorithm.loc[i,'F']) <= 0 and int(algorithm.loc[i,'M']) <= 0 and (int(algorithm.loc[i,'R'])+int(algorithm.loc[i,'F'])+int(algorithm.loc[i,'M'])) < 0:
            algorithm.loc[i,'Movement'] = "dropped"
        elif int(algorithm.loc[i,'R']) == 0 and int(algorithm.loc[i,'F']) == 0 and int(algorithm.loc[i,'M']) == 0 and (int(algorithm.loc[i,'R'])+int(algorithm.loc[i,'F'])+int(algorithm.loc[i,'M'])) == 0:
            algorithm.loc[i,'Movement'] = "stayed"
        elif (int(algorithm.loc[i,'R'])+int(algorithm.loc[i,'F'])+int(algorithm.loc[i,'M'])) > 0:
            algorithm.loc[i,'Movement'] = "marginally gained"
        elif (int(algorithm.loc[i,'R'])+int(algorithm.loc[i,'F'])+int(algorithm.loc[i,'M'])) < 0:
            algorithm.loc[i,'Movement'] = "marginally dropped"
    else:
        algorithm.loc[i,'Movement'] = "ambiguous"

for i in range(len(algorithm)):
    if algorithm.loc[i,'Movement'] == 0 or algorithm.loc[i,'Movement'] == '':
        algorithm.loc[i,'Movement'] = "ambiguous"


In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n5. RFM analysis processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 5. Demo RFM Data is in process: ProgressBar:10%")


In [ ]:
def add_prevaum(segdf,rundat,num_mon):
    from datetime import date
    from dateutil.relativedelta import relativedelta
    mths_ago = eom(currdt,num_mon)
    rundat = rundat[rundat['date']==mths_ago][['CGI','aum']]
    rundat.columns = ['CGI','aum_'+str(num_mon)]
    segdf = merge_dt(segdf,rundat)
    return segdf

def merge_dt(a,b):
    a = pd.merge(a,b,how='left',on='CGI')
    return a

In [ ]:
########################### Preparing historic asset data
cgi_master = contacts_base.copy()
cgi_master.columns=['CGI']
assets = df_assets_master.groupby(['Contact_Global_ID','Month']).agg({'new_aum':'sum'}).reset_index()
assets.rename(columns={'Contact_Global_ID':'CGI','Month':'date','new_aum':'aum'},inplace=True)

#################### Rollup Assets Calculation 

df_assets = add_prevaum(cgi_master,assets[['CGI','aum','date']],0)
df_assets = add_prevaum(df_assets,assets[['CGI','aum','date']],3)
df_assets = add_prevaum(df_assets,assets[['CGI','aum','date']],6)
df_assets = add_prevaum(df_assets,assets[['CGI','aum','date']],9)
df_assets = add_prevaum(df_assets,assets[['CGI','aum','date']],12)
df_assets = add_prevaum(df_assets,assets[['CGI','aum','date']],15)
df_assets = add_prevaum(df_assets,assets[['CGI','aum','date']],18)
df_assets = add_prevaum(df_assets,assets[['CGI','aum','date']],21)
# df_assets = add_prevaum(df_assets,assets[['CGI','aum','date']],24)
X = np.array(range(0,8)) + 1
df_assets = df_assets.fillna(0)
## aum Growth
assets=df_assets.copy()
assets['aum_slope'] = [linregress(X,[a,b,c,d,e,f,g,h])[0] for a,b,c,d,e,f,g,h in zip(assets['aum_21'],assets['aum_18'],assets['aum_15'],assets['aum_12'],assets['aum_9'],assets['aum_6'],assets['aum_3'],assets['aum_0'])]
assets['aum_qtrlyavg'] = [np.mean([a,b,c,d,e,f,g,h]) for a,b,c,d,e,f,g,h in zip(assets['aum_0'],assets['aum_3'],assets['aum_6'],assets['aum_9'],assets['aum_12'],assets['aum_15'],assets['aum_18'],assets['aum_21'])]

In [ ]:
#################### Outlier Treatment at Transaction level  ###
sales_original = sales.copy()
limit = sales[['Alloc_Sales_Period']].describe([0.995]).iloc[5,:]
sales["sales"] = sales[['Alloc_Sales_Period']].clip(upper=limit,axis=1)
sales=sales[['Contact_Global_ID','Process_Date','sales']]
sales.columns = ['CGI', 'date','sales']
sales = sales.groupby(['date','CGI']).agg({'sales':'sum'}).reset_index()

In [ ]:
##### Rollup sales and frequencies 
##################################

rollup_3monsales = pd.DataFrame(sales[sales['date'] > eom(currdt,3)].groupby(['CGI'])['sales'].sum()).reset_index()
rollup_3monsales.columns = ['CGI','rollup_3monsales']

rollup_3_6monsales = pd.DataFrame(sales[(sales['date'] > eom(currdt,9)) & (sales['date'] <= eom(currdt,3))].groupby(['CGI'])['sales'].sum()).reset_index()
rollup_3_6monsales.columns = ['CGI','rollup_3_6monsales']

rollup_12monsales = pd.DataFrame(sales[sales['date'] > eom(currdt,12)].groupby(['CGI'])['sales'].sum()).reset_index()
rollup_12monsales.columns = ['CGI','rollup_12monsales']

rollup_12_24monsales = pd.DataFrame(sales[(sales['date']> eom(currdt,24)) & (sales['date'] <= eom(currdt,12))].groupby(['CGI'])['sales'].sum()).reset_index()
rollup_12_24monsales.columns = ['CGI','rollup_12_24monsales']

rollup_3monfreq = pd.DataFrame(sales[(sales['date'] > eom(currdt,3)) & (sales['sales']>=0)].groupby(['CGI'])['date'].count()).reset_index()
rollup_3monfreq.columns = ['CGI','rollup_3monfreq']

rollup_3_6monfreq = pd.DataFrame(sales[(sales['date'] > eom(currdt,9)) & (sales['date'] <= eom(currdt,3)) & (sales['sales']>=0)].groupby(['CGI'])['date'].count()).reset_index()
rollup_3_6monfreq.columns = ['CGI','rollup_3_6monfreq']

rollup_12monfreq = pd.DataFrame(sales[(sales['date'] > eom(currdt,12)) & (sales['sales']>=0)].groupby(['CGI'])['date'].count()).reset_index()
rollup_12monfreq.columns = ['CGI','rollup_12monfreq']


###Count of Months where sales greater than 1000
#####################################################
SalesGT1000 = sales[(sales['date'] > eom(currdt,12))]
# SalesGT1000['date'] = SalesGT1000['date'].apply(lambda x:x.date())
SalesGT1000['date'] = SalesGT1000['date'].apply(lambda x:(x.year,x.month,1))
SalesGT1000 = SalesGT1000.groupby(['CGI','date']).sales.sum().reset_index()
SalesGT1000 = SalesGT1000[SalesGT1000['sales']>=1000]
SalesGT1000 = SalesGT1000.groupby(['CGI'])['date'].count().reset_index()
SalesGT1000.columns = ['CGI','SalesGT1000']


#### Count of months where sales is Zero in last 12 months 
#############################################################
Sales0_in12mon = pd.DataFrame(sales[(sales['date'] > eom(currdt,12)) & (sales['sales']==0)].groupby(['CGI'])['date'].count()).reset_index()
Sales0_in12mon.columns = ['CGI','Sales0_in12mon']

assets = merge_dt(assets,rollup_3monsales)
assets = merge_dt(assets,rollup_3_6monsales)
assets = merge_dt(assets,rollup_3monfreq)
assets = merge_dt(assets,rollup_3_6monfreq)
assets = merge_dt(assets,rollup_12monfreq)
assets = merge_dt(assets,rollup_12monsales)
assets = merge_dt(assets,rollup_12_24monsales)
assets = merge_dt(assets,SalesGT1000)
assets = merge_dt(assets,Sales0_in12mon)

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n6. sales based factors processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 6. Demo Sales based factors is in process: ProgressBar:30%")


In [ ]:
redemptions=redemptions[['Contact_Global_ID', 'Process_Date','Redemptions']]
redemptions.columns=['CGI', 'date', 'redemptions']

################ Rollup Redemptions
rollup_12monredemptions = pd.DataFrame(redemptions[(redemptions['date'] > eom(currdt,12))].groupby(['CGI'])['redemptions'].sum()).reset_index()
rollup_12monredemptions.columns = ['CGI','rollup_12monredemptions']

rollup_12_24monredemptions = pd.DataFrame(redemptions[(redemptions['date'] > eom(currdt,24)) & (redemptions['date'] <= eom(currdt,12))].groupby(['CGI'])['redemptions'].sum()).reset_index()
rollup_12_24monredemptions.columns = ['CGI','rollup_12_24monredemptions']

assets = merge_dt(assets,rollup_12monredemptions)
assets = merge_dt(assets,rollup_12_24monredemptions)

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n7. redemptions based factors processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 7. Demo redemptions based factors is in process: ProgressBar:30%")


In [ ]:
################## 12 month moving average of # of sales
def add_prevfreq(segdf,rundat,num_mon):
    from datetime import date
    from dateutil.relativedelta import relativedelta
    mths_ago = eom(currdt,num_mon)
    twelvemthsago = mths_ago - relativedelta(months=12)
    rundat = pd.DataFrame(rundat[(rundat['date'] > twelvemthsago) & (rundat['date'] <= mths_ago) & (rundat['sales']>0)].groupby(['CGI'])['date'].count()).reset_index()
    rundat['date'] = [a/12 for a in rundat['date']]
    rundat.columns = ['CGI','last_12monfreqavg_'+str(num_mon)]
    segdf = merge_dt(segdf,rundat)
    return segdf

assets = add_prevfreq(assets,sales[['CGI','sales','date']],1)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],2)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],3)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],4)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],5)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],6)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],7)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],8)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],9)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],10)
assets = add_prevfreq(assets,sales[['CGI','sales','date']],11)

##################### 12 month moving average of sales
def add_prevsales(segdf,rundat,num_mon):
    from datetime import date
    from dateutil.relativedelta import relativedelta
    mths_ago = eom(currdt,num_mon)
    twelvemthsago = mths_ago - relativedelta(months=12)
    rundat = pd.DataFrame(rundat[(rundat['date'] > twelvemthsago) & (rundat['date'] <= mths_ago)].groupby(['CGI'])['sales'].sum()).reset_index()
    rundat['sales'] = [a/12 for a in rundat['sales']]
    rundat.columns = ['CGI','last_12monsalesavg_'+str(num_mon)]
    segdf = merge_dt(segdf,rundat)
    return segdf
#################### 3 month moving average  
def add_threeprevsales(segdf,rundat,num_mon):
    from datetime import date
    from dateutil.relativedelta import relativedelta
    mths_ago = eom(currdt,num_mon)
    threemthsago = mths_ago - relativedelta(months=3)
    rundat = pd.DataFrame(rundat[(rundat['date'] > threemthsago) & (rundat['date'] <= mths_ago)].groupby(['CGI'])['sales'].sum()).reset_index()
    rundat['sales'] = [a/3 for a in rundat['sales']]
    rundat.columns = ['CGI','last_3monsalesavg_'+str(num_mon)]
    segdf = merge_dt(segdf,rundat)
    return segdf


####################### 12 months Lag variable for sales to calculate slope
assets = add_prevsales(assets,sales[['CGI','sales','date']],1)
assets = add_prevsales(assets,sales[['CGI','sales','date']],2)
assets = add_prevsales(assets,sales[['CGI','sales','date']],3)
assets = add_prevsales(assets,sales[['CGI','sales','date']],4)
assets = add_prevsales(assets,sales[['CGI','sales','date']],5)
assets = add_prevsales(assets,sales[['CGI','sales','date']],6)
assets = add_prevsales(assets,sales[['CGI','sales','date']],7)
assets = add_prevsales(assets,sales[['CGI','sales','date']],8)
assets = add_prevsales(assets,sales[['CGI','sales','date']],9)
assets = add_prevsales(assets,sales[['CGI','sales','date']],10)
assets = add_prevsales(assets,sales[['CGI','sales','date']],11)



################ 3 months Lag variable for sales to calculate slope
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],1)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],2)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],3)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],4)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],5)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],6)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],7)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],8)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],9)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],10)
assets = add_threeprevsales(assets,sales[['CGI','sales','date']],11)

assets.fillna(0,inplace=True)

In [ ]:
############# projected sales based on 12 month moving average
assets['last_12mon_monthly_avg_sales'] = [a/12 for a in assets['rollup_12monsales']]

X = np.array(range(0,12)) + 1
assets['salesslope'] = [linregress(X,[a,b,c,d,e,f,g,h,j,k,l,m])[0] for a,b,c,d,e,f,g,h,j,k,l,m in zip(assets['last_12monsalesavg_11'],assets['last_12monsalesavg_10'],assets['last_12monsalesavg_9'],assets['last_12monsalesavg_8'],assets['last_12monsalesavg_7'],assets['last_12monsalesavg_6'],assets['last_12monsalesavg_5'],assets['last_12monsalesavg_4'],assets['last_12monsalesavg_3'],assets['last_12monsalesavg_2'],assets['last_12monsalesavg_1'],assets['last_12mon_monthly_avg_sales'])]


assets['proj_sales'] = [(x+y)*12 for x,y in zip(assets['salesslope'],assets['last_12mon_monthly_avg_sales'])]
assets['proj_sales'] = [0 if a<0 else a for a in assets['proj_sales']]

assets['salesslope_byavg'] = [a/b if b!=0 else 0 for a,b in zip(assets['salesslope'],assets['last_12mon_monthly_avg_sales'])]

del assets['last_12monsalesavg_11'],assets['last_12monsalesavg_10'],assets['last_12monsalesavg_9'],assets['last_12monsalesavg_8'],assets['last_12monsalesavg_7'],assets['last_12monsalesavg_6'],assets['last_12monsalesavg_5'],assets['last_12monsalesavg_4'],assets['last_12monsalesavg_3'],assets['last_12monsalesavg_2'],assets['last_12monsalesavg_1']

In [ ]:
assets['prev_3monthsales'] = [a/3 for a in assets['rollup_3_6monsales']]
assets['prev_3monthfreq'] = [a/3 for a in assets['rollup_3_6monfreq']]
assets['avg_historical_12monsales'] = [(a+b)/2 for a,b in zip(assets['rollup_12monsales'],assets['rollup_12_24monsales'])]    
assets['max_historical_12monsales'] = [max(a,b) for a,b in zip(assets['rollup_12monsales'],assets['rollup_12_24monsales'])]   
assets['netsales12m'] = [(a+b) for a,b in zip(assets['rollup_12monsales'],assets['rollup_12monredemptions'])]
assets['netsales12_24m'] = [(a+b) for a,b in zip(assets['rollup_12_24monsales'],assets['rollup_12_24monredemptions'])]


assets = assets.fillna(0)
assets['sales_recent_freqgrowth'] = ['Yes' if int(a)>int(b/2) else 'No' if int(a)<int(b/2) else 'Equal' for a,b in zip(assets['rollup_3monfreq'],assets['rollup_3_6monfreq'])]
assets['sales_recent_growth'] = ['Yes' if int(a)>int(b/2) else 'No' if int(a)<int(b/2) else 'Equal' for a,b in zip(assets['rollup_3monsales'],assets['rollup_3_6monsales'])]
assets['last_12_month_freq_avg'] = [a/12 for a in assets['rollup_12monfreq']]
X = np.array(range(0,12)) + 1
assets['salesfreqslope'] = [linregress(X,[a,b,c,d,e,f,g,h,j,k,l,m])[0] for a,b,c,d,e,f,g,h,j,k,l,m in zip(assets['last_12monfreqavg_11'],assets['last_12monfreqavg_10'],assets['last_12monfreqavg_9'],assets['last_12monfreqavg_8'],assets['last_12monfreqavg_7'],assets['last_12monfreqavg_6'],assets['last_12monfreqavg_5'],assets['last_12monfreqavg_4'],assets['last_12monfreqavg_3'],assets['last_12monfreqavg_2'],assets['last_12monfreqavg_1'],assets['last_12_month_freq_avg'])]
assets['salesfreqslope_byavg'] = [a/b if b!=0 else 0 for a,b in zip(assets['salesfreqslope'],assets['last_12_month_freq_avg'])]
assets['sales_recent_freqgrowth'] = [1 if a=='Yes' else 0 if a=='Equal' else -1 for a in assets['sales_recent_freqgrowth']]    
assets['sales_recent_growth'] = [1 if a=='Yes' else 0 if a=='Equal' else -1 for a in assets['sales_recent_growth']]    

assets['Sales_GT_100K_2years'] = [1 if (a>=100000) | (b>=100000)  else 0 for a,b in zip(assets['rollup_12monsales'], assets['rollup_12_24monsales'])]

assets['last_3mon_monthly_avg_sales'] = [a/3 for a in assets['rollup_3monsales']]

assets['max_aum_2years'] = [max(a,b,c,d,e,f,g,h) for a,b,c,d,e,f,g,h in zip(assets['aum_0'],assets['aum_3'],assets['aum_6'],assets['aum_9'],assets['aum_12'],assets['aum_15'],assets['aum_18'],assets['aum_21'])]

assets['AUM_GT_1M'] = [1 if a>=1000000 else 0 for a in assets['aum_0']]
assets['aumslope_byavg'] = [a/b if b>0 else 0 for a,b in zip(assets['aum_slope'],assets['aum_qtrlyavg'])]



In [ ]:
############### projected sales based on 3m moving average
X = np.array(range(0,12)) + 1
assets['salesslope_3mon'] = [linregress(X,[a,b,c,d,e,f,g,h,j,k,l,m])[0] for a,b,c,d,e,f,g,h,j,k,l,m in zip(assets['last_3monsalesavg_11'],assets['last_3monsalesavg_10'],assets['last_3monsalesavg_9'],assets['last_3monsalesavg_8'],assets['last_3monsalesavg_7'],assets['last_3monsalesavg_6'],assets['last_3monsalesavg_5'],assets['last_3monsalesavg_4'],assets['last_3monsalesavg_3'],assets['last_3monsalesavg_2'],assets['last_3monsalesavg_1'],assets['last_3mon_monthly_avg_sales'])]

assets['proj_sales_3mon'] = [(x+y)*12 for x,y in zip(assets['salesslope_3mon'],assets['last_3mon_monthly_avg_sales'])]
assets['proj_sales_3mon'] = [0 if a<0 else a for a in assets['proj_sales_3mon']]

assets['salesslope_byavg_3mon'] = [a/b if b!=0 else 0 for a,b in zip(assets['salesslope_3mon'],assets['last_3mon_monthly_avg_sales'])]

del assets['last_3monsalesavg_11'],assets['last_3monsalesavg_10'],assets['last_3monsalesavg_9'],assets['last_3monsalesavg_8'],assets['last_3monsalesavg_7'],assets['last_3monsalesavg_6'],assets['last_3monsalesavg_5'],assets['last_3monsalesavg_4'],assets['last_3monsalesavg_3'],assets['last_3monsalesavg_2'],assets['last_3monsalesavg_1']

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n8. projected sales factors processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 8. Demo projected sales factors is in process: ProgressBar:40%")


In [ ]:
assets['nsales_segment'] = np.where((assets['netsales12m'] == 0) & (assets['netsales12_24m'] == 0), 'Holding Asset',
np.where((assets['netsales12m'] <= 0) & (assets['netsales12_24m'] <= 0), 'Redeeming',
np.where((assets['netsales12m'] > 0) & (assets['netsales12_24m'] > 0), 'Adding Asset',
np.where((assets['netsales12m'] > assets['netsales12_24m']), 'Recent Pickup',
np.where((assets['netsales12m'] < assets['netsales12_24m']), 'Recent Decline','Others')))))

assets['momentum'] = np.where((assets['proj_sales_3mon'] == 0) & (assets['proj_sales'] == 0), '0',
np.where((assets['proj_sales_3mon']*4 > assets['proj_sales']), 'Short Term Momentum',
np.where((assets['proj_sales_3mon']*4 < assets['proj_sales']), 'Long Term Momentum','Consistent')))

assets['max_historical_12monsales'] = [max(a,b) for a,b in zip(assets['rollup_12monsales'],assets['rollup_12_24monsales'])]
assets['max_AUM_2years'] = [max(a,b,c,d,e,f,g,h) for a,b,c,d,e,f,g,h in zip(assets['aum_0'],assets['aum_3'],assets['aum_6'],assets['aum_9'],assets['aum_12'],assets['aum_15'],assets['aum_18'],assets['aum_21'])]

assets['aumslope_byavg'] = [a/b if b>0 else 0 for a,b in zip(assets['aum_slope'],assets['aum_qtrlyavg'])]

# assets['Overall_Activity_DPM'] = [1 if ((a>0) | (b>0) | (c>0) | (d>0) | (e>0) | (f>0)) else 0 for a,b,c,d,e,f in
# zip(assets['rollup_12monactivity'], assets['rollup_12WebVisits'],
# assets['rollup_12mondownloads'], assets['rollup_12monclicks'], assets['Desgn_Award'],
# assets['Webinar_Attendee'])]


In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n9. netsales sales factors processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 9. Demo netsales sales factors is in process: ProgressBar:40%")


In [ ]:
####################### # funds based on latest AUM 
num_funds = df_assets_master[(df_assets_master['new_aum'] > 0)&(df_assets_master['asset_date_asof'] == currdt)]
fund_name_std=pd.read_csv('/dbfs/mnt/demo/dl_demo_ds/pipeline/dictionaries/fundname_std.csv')
fund_name_std.rename(columns={'fund_name':'FUND_NAME'},inplace=True)

num_funds= num_funds.merge(fund_name_std, on = 'FUND_NAME', how='left')
num_funds = num_funds[['Contact_Global_ID','Month','fund','new_aum']]
num_funds.columns = ['CGI','date','fund','aum']
num_funds = num_funds.groupby('CGI').agg({'fund':'nunique'}).reset_index()
num_funds.columns = ['CGI','num_funds_assets']

assets = pd.merge(assets,num_funds,how='left',on='CGI')
assets['num_funds_assets'].fillna(0,inplace=True)

############## # of funds based on last 12 months sales
num_funds = sales_original[(sales_original['Alloc_Sales_Period'] > 0) & (sales_original['Process_Date'] > eom(currdt,12)) & (sales_original['Process_Date'] <= currdt)]
num_funds= num_funds.merge(fund_name_std, on = 'FUND_NAME', how='left')
num_funds = num_funds[['Contact_Global_ID','Process_Date','fund', 'Alloc_Sales_Period']]
num_funds.reset_index(inplace=True,drop=True)
num_funds = num_funds.groupby('Contact_Global_ID').agg({'fund':'nunique'}).reset_index()
num_funds.columns = ['CGI','num_funds_sales']

assets = pd.merge(assets,num_funds,how='left',on='CGI')
assets['num_funds_sales'].fillna(0,inplace=True)

########################### sales for last 2 years
last12msales = sales_original[(sales_original['Alloc_Sales_Period'] > 0)&(sales_original['Process_Date'] >= eom(currdt,12)) & (sales_original['Process_Date'] <= currdt)]
last12msales = last12msales[['Contact_Global_ID','Process_Date','Alloc_Sales_Period']]
last12msales.columns = ['CGI','date','sales']
last12msales = last12msales.groupby('CGI').sales.sum().reset_index()
last12msales.columns = ['CGI','last12m_sales']

assets = pd.merge(assets,last12msales,how='left',on='CGI')
assets['last12m_sales'].fillna(0,inplace=True)

#filter last 12-24m yrs of data
last1224msales = sales_original[(sales_original['Alloc_Sales_Period'] > 0)& (sales_original['Process_Date'] >= eom(currdt,24)) & (sales_original['Process_Date'] <= eom(currdt,12))]
last1224msales = last1224msales[['Contact_Global_ID','Process_Date','Alloc_Sales_Period']]
last1224msales.columns = ['CGI','date','sales']
last1224msales = last1224msales.groupby('CGI').sales.sum().reset_index()
last1224msales.columns = ['CGI','last12_24m_sales']

assets = pd.merge(assets,last1224msales,how='left',on='CGI')
assets['last12_24m_sales'].fillna(0,inplace=True)

del num_funds
del last12msales
del last1224msales

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n10. # funds and historic sales factors processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 10. Demo # funds and historic sales factors is in process: ProgressBar:50%")


In [ ]:
######################### read activities data
activity = activity.dropna(subset=['Contact_Global_ID'])
activity['Activity_Mode'] = np.where(activity['ACTIVITY_TYPE_NORM'].isin(['Phone Call','Video Call']), "CALL", np.where(activity['ACTIVITY_TYPE_NORM'].isin(['In Person']), "VISIT", activity['ACTIVITY_TYPE_NORM']))

Contact_3M_data = activity[(activity['ACTIVITY_START_DATE'] > currdt - relativedelta(months=3))& (activity['ACTIVITY_START_DATE'] <= currdt)]
Contact_3M_data = Contact_3M_data.groupby(['Contact_Global_ID','Activity_Mode']).ACTIVITY_TYPE_NORM.count().reset_index()
Contact_3M_data = pd.pivot_table(data=Contact_3M_data,index='Contact_Global_ID',columns='Activity_Mode',values='ACTIVITY_TYPE_NORM',aggfunc='sum').reset_index()
Contact_3M_data.fillna(0,inplace=True)
Contact_3M_data.set_index('Contact_Global_ID',inplace=True)
Contact_3M_data.columns = Contact_3M_data.columns + '_3m'
Contact_3M_data.reset_index(drop=False, inplace=True)
Contact_3M_data.rename(columns={'Contact_Global_ID':'CGI'},inplace=True)
Contact_3M_data = Contact_3M_data[['CGI','CALL_3m','VISIT_3m']]

Contact_12M_data = activity[(activity['ACTIVITY_START_DATE'] > currdt - relativedelta(months=12))& (activity['ACTIVITY_START_DATE'] <= currdt)]
Contact_12M_data = Contact_12M_data.groupby(['Contact_Global_ID','Activity_Mode']).ACTIVITY_TYPE_NORM.count().reset_index()
Contact_12M_data = pd.pivot_table(data=Contact_12M_data,index='Contact_Global_ID',columns='Activity_Mode',values='ACTIVITY_TYPE_NORM',aggfunc='sum').reset_index()
Contact_12M_data.fillna(0,inplace=True)
Contact_12M_data.set_index('Contact_Global_ID',inplace=True)
Contact_12M_data.columns = Contact_12M_data.columns + '_12m'
Contact_12M_data.reset_index(drop=False, inplace=True)
Contact_12M_data.rename(columns={'Contact_Global_ID':'CGI'},inplace=True)
Contact_12M_data = Contact_12M_data[['CGI','CALL_12m','VISIT_12m']]

all_activity = pd.merge(Contact_3M_data,Contact_12M_data,how='outer',on='CGI')
all_activity['Total_activity'] = all_activity['CALL_12m'] +	all_activity['VISIT_12m']

assets = pd.merge(assets,all_activity,how='left',on='CGI')
assets.fillna(0,inplace=True)

del Contact_3M_data
del Contact_12M_data
del all_activity

In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n11. activity factors processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 11. Demo activity factors is in process: ProgressBar:60%")


In [ ]:
#################### fund returns data filter for latest date in sql query
data_returns = spark.sql(""" SELECT * FROM sch_demo_07.table_10 where country_code = 'XX1' and share_class_code = 'Z' and currency_code = 'USD' and as_of_date_std  == '{0}' """.format(str(currdt)))
# data_returns_v2 = data_returns_v2.select('*', to_date(data_returns_v2.as_of_date_std).alias('date'))
# data_returns_v2 = data_returns_v2[data_returns_v2.date > (currdt - relativedelta(years=2))]
data_returns = data_returns.toPandas()
data_returns['as_of_date_std'] = data_returns['as_of_date_std'].apply(lambda x:datetime.datetime.strptime(x, '%Y-%m-%d').date())
data_returns['fund_name']=data_returns['fund_name'].str.upper()

############### standardize fund name in peer ranking table
lookup = pd.read_csv('/dbfs/mnt/demo/dl_demo_ds/pipeline/dictionaries/fund_lookup.csv')
lookup.columns=['fund_name','fund']
lookup['fund_name']=lookup['fund_name'].str.upper()
lookup['fund']=lookup['fund'].str.upper()

data_returns= data_returns.merge(lookup, left_on = 'fund_name',right_on = 'fund_name', how='left')

Domicile_A = data_returns[data_returns.domicile == 'Domicile_A']
non_Domicile_A = data_returns[data_returns.domicile != 'Domicile_A']
non_Domicile_A = non_Domicile_A[~non_Domicile_A.fund.isin(Domicile_A.fund)]
data_returns = pd.concat([Domicile_A.reset_index(drop=True),non_Domicile_A.reset_index(drop=True)],axis=0)

################### Returns Clustering
from sklearn.cluster import KMeans
data_returns.fund_cumulative_ret_1yr = data_returns.fund_cumulative_ret_1yr.astype('float')
data_returns[['fund_cumulative_ret_1yr']].fillna(0,inplace=True)
data_returns = data_returns[data_returns.fund_cumulative_ret_1yr.isna()==False]

kmeans = KMeans(n_clusters=3, init='k-means++',random_state=12345)
kmeans.fit(data_returns[['fund_cumulative_ret_1yr']])
pred = kmeans.predict(data_returns[['fund_cumulative_ret_1yr']])
data_returns['cluster']=pred

############### cluster name
rankcluster = data_returns.groupby(['cluster']).agg({'fund_cumulative_ret_1yr':'mean'}).reset_index()
rankcluster['cluster_rank'] = rankcluster.fund_cumulative_ret_1yr.rank(axis=0, method='first',ascending=True, pct=False)
rankcluster['cluster_name'] = np.where((rankcluster.cluster_rank == 1),'low_Returns',np.where((rankcluster.cluster_rank == 2),'med_Returns', np.where((rankcluster.cluster_rank == 3),'high_Returns',None)))

data_returns = data_returns.merge(rankcluster[['cluster','cluster_name']], on = 'cluster',how='left')
data_returns = data_returns[['as_of_date_std','fund_cumulative_ret_1yr','fund','cluster_name']]
data_returns['fund']=data_returns['fund'].str.upper()
data_returns['fund'] = data_returns['fund'].str.replace(' +', ' ')
data_returns['fund'] = data_returns['fund'].str.strip()
data_returns.rename(columns={'cluster_name':'cluster'},inplace=True)


In [ ]:
fund_name_std=pd.read_csv('/dbfs/mnt/demo/dl_demo_ds/pipeline/dictionaries/fundname_std.csv')
fund_name_std.rename(columns={'fund_name':'FUND_NAME'},inplace=True)
asset_class=pd.read_csv('/dbfs/mnt/demo/dl_demo_ds/pipeline/dictionaries/asset_class.csv')

funds_return = df_assets_master[df_assets_master.asset_date_asof == currdt]
funds_return = funds_return.merge(fund_name_std,on='FUND_NAME',how='left')
funds_return = funds_return.merge(data_returns[['fund','cluster','fund_cumulative_ret_1yr']], on='fund',how='left')
funds_return = funds_return.merge(asset_class[['fund', 'FUND_OBJECTIVES']],on='fund',how='left')
funds_return.rename(columns={'Contact_Global_ID':'CGI'},inplace=True)

#################### # funds in High medium and low performing segments
funds_return1 = funds_return.groupby(['CGI','cluster','fund']).agg({'new_aum':'sum'}).reset_index()
funds_return1 = pd.pivot_table(funds_return1, values=['fund'],index = ['CGI'], columns='cluster' , aggfunc = 'count' ).reset_index()
funds_return1.set_index('CGI',inplace=True)
funds_return1.columns =[s1 + '_'+ str(s2) for (s1,s2) in funds_return1.columns.tolist()]
funds_return1.reset_index(drop=False,inplace=True)
funds_return1.fillna(0,inplace=True)

################ average fund returns
funds_return2 = funds_return[~funds_return.fund_cumulative_ret_1yr.isna()]
funds_return2 = funds_return2.groupby(['CGI']).agg({'fund_cumulative_ret_1yr':'mean'}).reset_index()
funds_return2.rename(columns={'fund_cumulative_ret_1yr':'Fund_Returns'},inplace=True)

funds_return_vars = funds_return1.merge(funds_return2,on='CGI',how='outer')
del funds_return1
del funds_return2

############# returns average at asset class level
funds_return.fund_cumulative_ret_1yr=funds_return.fund_cumulative_ret_1yr.astype(float)
funds_return.fund_cumulative_ret_1yr.fillna(0,inplace=True)
funds_return.FUND_OBJECTIVES.fillna('Others',inplace=True)

pv_returns = pd.pivot_table(funds_return, index = ['CGI'], columns =['FUND_OBJECTIVES'], values =['fund_cumulative_ret_1yr'], aggfunc = np.mean).reset_index()
pv_returns.set_index('CGI',inplace=True)
pv_returns.columns =[s1 + '_'+ str(s2) for (s1,s2) in pv_returns.columns.tolist()]
pv_returns.reset_index(drop=False,inplace=True)
pv_returns.fillna(0,inplace=True)

######## mean aum in each asset class
pv_assets=pd.pivot_table(funds_return, index = ['CGI'], columns =['FUND_OBJECTIVES'], values =['new_aum'], aggfunc = np.mean).reset_index()
pv_assets.rename(columns={'new_aum':'aum'},inplace=True)
pv_assets.set_index('CGI',inplace=True)
pv_assets.columns =[s1 + '_'+ str(s2) for (s1,s2) in pv_assets.columns.tolist()]
pv_assets.reset_index(drop=False,inplace=True)
pv_assets.fillna(0,inplace=True)

pv_returns_assets = pv_returns.merge(pv_assets,on='CGI',how='outer')
funds_return_vars = funds_return_vars.merge(pv_returns_assets,on='CGI',how='outer')

del pv_returns
del pv_assets

assets = assets.merge(funds_return_vars,on='CGI',how='left')
assets.fillna(0,inplace=True)


In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n12. returns based factors processed: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 12. Demo returns based factors is in process: ProgressBar:70%")


In [ ]:
##################### take only M variable from RFM table
algorithm.rename(columns={'Contact_Global_ID_new':'CGI'},inplace=True)
assets = pd.merge(assets,algorithm,how='left',on='CGI')
assets['Movement'].fillna('ambiguous',inplace=True)
assets.RFM_f6.fillna(111,inplace=True)	
assets.RFM_s6.fillna(111,inplace=True)
assets['RFM_f6'] = assets['RFM_f6'].astype('int')
assets['RFM_s6'] = assets['RFM_f6'].astype('int')
assets.R.fillna(0,inplace=True)	
assets.F.fillna(0,inplace=True)
assets.M.fillna(0,inplace=True)


In [ ]:
assets['active_dormant'] = np.where((assets['rollup_12monsales'] > 0) | (assets['aum_0'] > 0),'active','dormant')

################ get last24m sales and last24 redemptions
assets['last24msales'] = assets['rollup_12monsales'] + assets['rollup_12_24monsales']
assets['last24mredemptions'] = assets['rollup_12monredemptions'] + assets['rollup_12_24monredemptions']


In [ ]:
batch_start = datetime.datetime.now(IST)
log_1 = '\n13. Dataprep is done: {0}\n'.format(str(batch_start)[0:19])
script_log.append(log_1)
pd.DataFrame(script_log).to_csv(Log_path+"/Demo_Dataprep_Logs.txt",index=False,columns=None)
print(" 13. Demo Dataprep final factors are in process: ProgressBar:70%")


In [ ]:
filename = 'Demo_Leads_' + str(currdt.strftime('%d%b%Y').upper()) + '.csv'
assets.to_csv(base_path +'/Demo Input/' +filename , index=False)
